# 原生 ROS 底盘：独立里程计与控制

连接原生轮速与 IMU，可选择 2D ICP；本例仅发送零速。配置和测量外参见 `native_chassis.example.yaml`。ROS 节点已运行时使用 `start_driver: false`。应用客户端 Notebook 为 `commands.ipynb`。

In [ ]:
from pathlib import Path
import os

root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "pyproject.toml").is_file() and (p / "rsim").is_dir())
assets = root / "assets"
assets.mkdir(exist_ok=True)
os.environ["NOTEBOOK_ASSETS_ROOT"] = str(assets)
config_file = root / "configs" / "chassis_ros2.yaml"

将模板复制到配置路径，填写话题与安装 Pose。`estimator: wheel_imu` 使用轮速与完整姿态 IMU；`scan_imu` 加入 RTAB ICP。所有 native 进程和协程由同一个 Runtime 管理。

In [ ]:
from rsim.config import load_chassis
from rsim.runtime import Runtime

robot = load_chassis(config_file, motion_enabled=False)
async with Runtime(robot.control, robot.odometry.status, robot.chassis.state):
    frame = await robot.pose.get(timeout=30)
    print(frame.data)
    print((await robot.odometry.status.get(timeout=3)).data)
    await robot.control.move(0.)
    await robot.control.rotate(yaw_deg=0.)
    print((await robot.chassis.state.get(timeout=3)).data)

对外发布时可直接组合 pose 与速度 sink；设备细节留在驱动进程。下面服务阻塞等待，Notebook 中断会释放 Runtime 和原生资源。

In [ ]:
from rsim.drivers import Chassis

# Run this cell to host the service until interrupted.
# robot = load_chassis(config_file, motion_enabled=False)
# service = Chassis(pose=robot.pose, velocity=robot.velocity,
#                   state=robot.chassis.state, name="chassis")
# async with Runtime(service):
#     await service.wait()